# 2026 World Cup Prediction Extension

This notebook applies our trained logistic regression model to a simulated 2026 FIFA World Cup bracket.

The main project trained and evaluated the model on historical international soccer matches. This notebook is an extension that uses the trained model to predict future World Cup matchups.

The model predicts one of three outcomes from the perspective of the listed home team:

- `home_win`
- `draw`
- `away_win`

For knockout rounds, draws are not valid final outcomes. Therefore, if the model predicts a draw in a knockout match, the higher-ranked team advances.

## Imports and Load Saved Model

The trained logistic regression model, scaler, and feature list were saved from the predictive modeling notebook. Loading the saved model prevents us from rerunning the full modeling workflow.

In [1]:
import joblib
import pandas as pd

# Load trained model, scaler, and feature list
model_bundle = joblib.load("../models/logistic_regression_model.joblib")

log_reg_model = model_bundle["model"]
scaler = model_bundle["scaler"]
features = model_bundle["features"]

print("Model loaded successfully.")
print("Model features:", features)

Model loaded successfully.
Model features: ['home_recent_win_rate', 'away_recent_win_rate', 'home_recent_goal_diff', 'away_recent_goal_diff', 'home_rank', 'away_rank', 'rank_difference', 'neutral']


## Load 2026 World Cup Team Features

We manually created a team-level CSV for the 48 World Cup teams. Each row contains:

- Team name
- FIFA ranking
- Latest win rate
- Latest average goal differential

This team-level dataset is later merged into fixture-level matchups to create the same model features used during training.

In [2]:
wc_features = pd.read_csv("../data/world_cup_2026/world_cup_2026_features.csv")

# Clean data types
wc_features["fifa_rank"] = wc_features["fifa_rank"].astype(int)
wc_features["latest_win_rate"] = wc_features["latest_win_rate"].astype(float)
wc_features["latest_goal_diff"] = wc_features["latest_goal_diff"].astype(float)

print("World Cup team features shape:", wc_features.shape)
display(wc_features.head())

print("\nMissing values:")
display(wc_features.isna().sum())

World Cup team features shape: (48, 5)


,team,ranking_name,fifa_rank,latest_win_rate,latest_goal_diff
0,France,France,1,0.8,1.8
1,Spain,Spain,2,0.6,2.2
2,Argentina,Argentina,3,1.0,3.0
3,England,England,4,0.6,1.6
4,Portugal,Portugal,5,0.4,1.6



Missing values:


team                0
ranking_name        0
fifa_rank           0
latest_win_rate     0
latest_goal_diff    0
dtype: int64

## Model Features

The trained model expects exactly these eight features:

- `home_recent_win_rate`
- `away_recent_win_rate`
- `home_recent_goal_diff`
- `away_recent_goal_diff`
- `home_rank`
- `away_rank`
- `rank_difference`
- `neutral`

Columns such as team names, group labels, and match numbers are only kept for readability. They are not passed into the model.

## Helper Functions

The following functions convert fixture-level matchups into model-ready prediction data.

They merge team-level features for the home and away teams, calculate rank difference, scale the features, generate model predictions, and save the results.

For knockout rounds, if the model predicts a draw, the higher-ranked team advances.

In [3]:
def build_prediction_input(fixtures_df, wc_features):
    """
    Merge team-level World Cup features into fixture-level matchups.
    Returns a dataframe with the exact model features needed for prediction.
    """
    home_features = wc_features[
        ["team", "fifa_rank", "latest_win_rate", "latest_goal_diff"]
    ].copy()

    home_features = home_features.rename(
        columns={
            "team": "home_team",
            "fifa_rank": "home_rank",
            "latest_win_rate": "home_recent_win_rate",
            "latest_goal_diff": "home_recent_goal_diff"
        }
    )

    away_features = wc_features[
        ["team", "fifa_rank", "latest_win_rate", "latest_goal_diff"]
    ].copy()

    away_features = away_features.rename(
        columns={
            "team": "away_team",
            "fifa_rank": "away_rank",
            "latest_win_rate": "away_recent_win_rate",
            "latest_goal_diff": "away_recent_goal_diff"
        }
    )

    prediction_input = fixtures_df.merge(
        home_features,
        on="home_team",
        how="left"
    )

    prediction_input = prediction_input.merge(
        away_features,
        on="away_team",
        how="left"
    )

    # Positive rank_difference means the home team is ranked better.
    # Example: home_rank = 2, away_rank = 20 -> rank_difference = 18
    prediction_input["rank_difference"] = (
        prediction_input["away_rank"] - prediction_input["home_rank"]
    )

    return prediction_input


def validate_prediction_input(prediction_input):
    """
    Check that all required model features are present and non-missing.
    """
    required_features = [
        "home_recent_win_rate",
        "away_recent_win_rate",
        "home_recent_goal_diff",
        "away_recent_goal_diff",
        "home_rank",
        "away_rank",
        "rank_difference",
        "neutral"
    ]

    print("Shape:", prediction_input.shape)

    print("\nMissing values:")
    display(prediction_input.isna().sum())

    missing_rows = prediction_input[
        prediction_input[required_features].isna().any(axis=1)
    ]

    print("\nRows with missing model features:", missing_rows.shape[0])
    display(missing_rows)

    return missing_rows


def get_predicted_winner(row, knockout=False):
    """
    Convert model prediction into a winner label.
    
    Group stage:
    - Draws remain draws.
    
    Knockout rounds:
    - Draws are resolved by advancing the higher-ranked team.
    """
    if row["predicted_result"] == "home_win":
        return row["home_team"]

    if row["predicted_result"] == "away_win":
        return row["away_team"]

    if not knockout:
        return "Draw"

    # Knockout tiebreaker: lower FIFA rank number is better
    if row["home_rank"] < row["away_rank"]:
        return row["home_team"]
    else:
        return row["away_team"]


def predict_matches(fixtures_df, wc_features, output_path, knockout=False):
    """
    Build prediction input, run model predictions, add predicted winners,
    and save the result to CSV.
    """
    prediction_input = build_prediction_input(fixtures_df, wc_features)

    context_columns = [
        col for col in ["group", "match_number"]
        if col in prediction_input.columns
    ]

    prediction_input = prediction_input[
        context_columns +
        [
            "home_team",
            "away_team",
            "neutral",
            "home_recent_win_rate",
            "away_recent_win_rate",
            "home_recent_goal_diff",
            "away_recent_goal_diff",
            "home_rank",
            "away_rank",
            "rank_difference"
        ]
    ].copy()

    validate_prediction_input(prediction_input)

    X = prediction_input[features]
    X_scaled = scaler.transform(X)

    predictions = prediction_input.copy()
    predictions["predicted_result"] = log_reg_model.predict(X_scaled)

    predictions["predicted_winner"] = predictions.apply(
        lambda row: get_predicted_winner(row, knockout=knockout),
        axis=1
    )

    predictions.to_csv(output_path, index=False)

    print("Saved:", output_path)

    display_columns = context_columns + [
        "home_team",
        "away_team",
        "predicted_result",
        "predicted_winner",
        "home_rank",
        "away_rank"
    ]

    display(predictions[display_columns])

    return predictions

## Create Group Stage Fixtures

We create a fixture dataframe for all 72 group-stage matches.

The `group` column is kept only for organization. It is not used as a model feature.

In [4]:
groups = {
    "A": ["Mexico", "South Africa", "South Korea", "Czech Republic"],
    "B": ["Canada", "Switzerland", "Qatar", "Bosnia and Herzegovina"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Tunisia", "Sweden"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Norway", "Iraq"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"]
}

# Six matches for each four-team group
pairings = [
    (0, 1),
    (2, 3),
    (0, 2),
    (3, 1),
    (3, 0),
    (1, 2)
]

host_teams = ["Mexico", "United States", "Canada"]

fixtures = []

for group, teams in groups.items():
    for home_idx, away_idx in pairings:
        home_team = teams[home_idx]
        away_team = teams[away_idx]

        # If a host team is involved, list the host as home_team when possible.
        # This lets neutral = 0 represent possible host advantage.
        if away_team in host_teams and home_team not in host_teams:
            home_team, away_team = away_team, home_team

        neutral = 0 if home_team in host_teams else 1

        fixtures.append({
            "group": group,
            "home_team": home_team,
            "away_team": away_team,
            "neutral": neutral
        })

wc_group_fixtures = pd.DataFrame(fixtures)

wc_group_fixtures.to_csv(
    "../data/world_cup_2026/world_cup_2026_group_stage_fixtures.csv",
    index=False
)

print("Saved: ../data/world_cup_2026/world_cup_2026_group_stage_fixtures.csv")
print("Shape:", wc_group_fixtures.shape)

display(wc_group_fixtures.head(12))

Saved: ../data/world_cup_2026/world_cup_2026_group_stage_fixtures.csv
Shape: (72, 4)


,group,home_team,away_team,neutral
0,A,Mexico,South Africa,0
1,A,South Korea,Czech Republic,1
2,A,Mexico,South Korea,0
3,A,Czech Republic,South Africa,1
4,A,Mexico,Czech Republic,0
5,A,South Africa,South Korea,1
6,B,Canada,Switzerland,0
7,B,Qatar,Bosnia and Herzegovina,1
8,B,Canada,Qatar,0
9,B,Bosnia and Herzegovina,Switzerland,1


## Group Stage Predictions

We apply the trained model to all 72 group-stage matches. Since group-stage matches can end in draws, predicted draws are kept as draws.

In [5]:
group_stage_predictions = predict_matches(
    fixtures_df=wc_group_fixtures,
    wc_features=wc_features,
    output_path="../data/world_cup_2026/world_cup_2026_group_stage_predictions.csv",
    knockout=False
)

Shape: (72, 11)

Missing values:


group                    0
home_team                0
away_team                0
neutral                  0
home_recent_win_rate     0
away_recent_win_rate     0
home_recent_goal_diff    0
away_recent_goal_diff    0
home_rank                0
away_rank                0
rank_difference          0
dtype: int64


Rows with missing model features: 0


,group,home_team,away_team,neutral,home_recent_win_rate,away_recent_win_rate,home_recent_goal_diff,away_recent_goal_diff,home_rank,away_rank,rank_difference


Saved: ../data/world_cup_2026/world_cup_2026_group_stage_predictions.csv


,group,home_team,away_team,predicted_result,predicted_winner,home_rank,away_rank
0,A,Mexico,South Africa,home_win,Mexico,15,60
1,A,South Korea,Czech Republic,home_win,South Korea,25,41
2,A,Mexico,South Korea,home_win,Mexico,15,25
3,A,Czech Republic,South Africa,home_win,Czech Republic,41,60
4,A,Mexico,Czech Republic,home_win,Mexico,15,41
...,...,...,...,...,...,...,...
67,L,Ghana,Panama,away_win,Panama,74,33
68,L,England,Ghana,home_win,England,4,74
69,L,Panama,Croatia,away_win,Croatia,33,11
70,L,Panama,England,away_win,England,33,4


## Round of 32 Predictions

The Round of 32 matchups were manually entered based on the predicted group-stage results.

Knockout matches cannot end in draws, so predicted draws are resolved by advancing the higher-ranked team.

In [6]:
round_of_32_matches = [
    ("South Korea", "Switzerland"),
    ("Germany", "Scotland"),
    ("Netherlands", "Morocco"),
    ("Brazil", "Japan"),
    ("France", "Australia"),
    ("Ecuador", "Senegal"),
    ("Mexico", "Ivory Coast"),
    ("England", "DR Congo"),
    ("United States", "Austria"),
    ("Belgium", "Czech Republic"),
    ("Colombia", "Croatia"),
    ("Spain", "Algeria"),
    ("Canada", "Egypt"),
    ("Argentina", "Uruguay"),
    ("Portugal", "Norway"),
    ("Turkey", "Iran")
]

round_of_32_fixtures = []

for i, (home_team, away_team) in enumerate(round_of_32_matches, start=73):
    neutral = 0 if home_team in host_teams or away_team in host_teams else 1

    round_of_32_fixtures.append({
        "match_number": i,
        "home_team": home_team,
        "away_team": away_team,
        "neutral": neutral
    })

round_of_32_fixtures = pd.DataFrame(round_of_32_fixtures)

round_of_32_fixtures.to_csv(
    "../data/world_cup_2026/world_cup_2026_round_of_32_fixtures.csv",
    index=False
)

display(round_of_32_fixtures)

,match_number,home_team,away_team,neutral
0,73,South Korea,Switzerland,1
1,74,Germany,Scotland,1
2,75,Netherlands,Morocco,1
3,76,Brazil,Japan,1
4,77,France,Australia,1
5,78,Ecuador,Senegal,1
6,79,Mexico,Ivory Coast,0
7,80,England,DR Congo,1
8,81,United States,Austria,0
9,82,Belgium,Czech Republic,1


In [7]:
round_of_32_predictions = predict_matches(
    fixtures_df=round_of_32_fixtures,
    wc_features=wc_features,
    output_path="../data/world_cup_2026/world_cup_2026_round_of_32_predictions.csv",
    knockout=True
)

Shape: (16, 11)

Missing values:


match_number             0
home_team                0
away_team                0
neutral                  0
home_recent_win_rate     0
away_recent_win_rate     0
home_recent_goal_diff    0
away_recent_goal_diff    0
home_rank                0
away_rank                0
rank_difference          0
dtype: int64


Rows with missing model features: 0


,match_number,home_team,away_team,neutral,home_recent_win_rate,away_recent_win_rate,home_recent_goal_diff,away_recent_goal_diff,home_rank,away_rank,rank_difference


Saved: ../data/world_cup_2026/world_cup_2026_round_of_32_predictions.csv


,match_number,home_team,away_team,predicted_result,predicted_winner,home_rank,away_rank
0,73,South Korea,Switzerland,away_win,Switzerland,25,19
1,74,Germany,Scotland,home_win,Germany,10,43
2,75,Netherlands,Morocco,home_win,Netherlands,7,8
3,76,Brazil,Japan,home_win,Brazil,6,18
4,77,France,Australia,home_win,France,1,27
5,78,Ecuador,Senegal,home_win,Ecuador,23,14
6,79,Mexico,Ivory Coast,home_win,Mexico,15,34
7,80,England,DR Congo,home_win,England,4,46
8,81,United States,Austria,home_win,United States,16,24
9,82,Belgium,Czech Republic,home_win,Belgium,9,41


## Round of 16 Predictions

The Round of 16 matchups were manually entered based on the predicted Round of 32 winners.

In [8]:
round_of_16_matches = [
    ("Germany", "France"),
    ("Switzerland", "Netherlands"),
    ("Brazil", "Ecuador"),
    ("Mexico", "England"),
    ("Colombia", "Spain"),
    ("United States", "Belgium"),
    ("Argentina", "Turkey"),
    ("Canada", "Portugal")
]

round_of_16_fixtures = []

for i, (home_team, away_team) in enumerate(round_of_16_matches, start=89):
    neutral = 0 if home_team in host_teams or away_team in host_teams else 1

    round_of_16_fixtures.append({
        "match_number": i,
        "home_team": home_team,
        "away_team": away_team,
        "neutral": neutral
    })

round_of_16_fixtures = pd.DataFrame(round_of_16_fixtures)

round_of_16_fixtures.to_csv(
    "../data/world_cup_2026/world_cup_2026_round_of_16_fixtures.csv",
    index=False
)

display(round_of_16_fixtures)

,match_number,home_team,away_team,neutral
0,89,Germany,France,1
1,90,Switzerland,Netherlands,1
2,91,Brazil,Ecuador,1
3,92,Mexico,England,0
4,93,Colombia,Spain,1
5,94,United States,Belgium,0
6,95,Argentina,Turkey,1
7,96,Canada,Portugal,0


In [9]:
round_of_16_predictions = predict_matches(
    fixtures_df=round_of_16_fixtures,
    wc_features=wc_features,
    output_path="../data/world_cup_2026/world_cup_2026_round_of_16_predictions.csv",
    knockout=True
)

Shape: (8, 11)

Missing values:


match_number             0
home_team                0
away_team                0
neutral                  0
home_recent_win_rate     0
away_recent_win_rate     0
home_recent_goal_diff    0
away_recent_goal_diff    0
home_rank                0
away_rank                0
rank_difference          0
dtype: int64


Rows with missing model features: 0


,match_number,home_team,away_team,neutral,home_recent_win_rate,away_recent_win_rate,home_recent_goal_diff,away_recent_goal_diff,home_rank,away_rank,rank_difference


Saved: ../data/world_cup_2026/world_cup_2026_round_of_16_predictions.csv


,match_number,home_team,away_team,predicted_result,predicted_winner,home_rank,away_rank
0,89,Germany,France,away_win,France,10,1
1,90,Switzerland,Netherlands,away_win,Netherlands,19,7
2,91,Brazil,Ecuador,home_win,Brazil,6,23
3,92,Mexico,England,home_win,Mexico,15,4
4,93,Colombia,Spain,away_win,Spain,13,2
5,94,United States,Belgium,home_win,United States,16,9
6,95,Argentina,Turkey,home_win,Argentina,3,22
7,96,Canada,Portugal,home_win,Canada,30,5


## Quarter-Final Predictions

The quarter-final matchups were manually entered based on the predicted Round of 16 winners.

In [10]:
quarter_final_matches = [
    ("France", "Netherlands"),
    ("Spain", "United States"),
    ("Brazil", "Mexico"),
    ("Argentina", "Canada")
]

quarter_final_fixtures = []

for i, (home_team, away_team) in enumerate(quarter_final_matches, start=97):
    neutral = 0 if home_team in host_teams or away_team in host_teams else 1

    quarter_final_fixtures.append({
        "match_number": i,
        "home_team": home_team,
        "away_team": away_team,
        "neutral": neutral
    })

quarter_final_fixtures = pd.DataFrame(quarter_final_fixtures)

quarter_final_fixtures.to_csv(
    "../data/world_cup_2026/world_cup_2026_quarter_final_fixtures.csv",
    index=False
)

display(quarter_final_fixtures)

,match_number,home_team,away_team,neutral
0,97,France,Netherlands,1
1,98,Spain,United States,0
2,99,Brazil,Mexico,0
3,100,Argentina,Canada,0


In [11]:
quarter_final_predictions = predict_matches(
    fixtures_df=quarter_final_fixtures,
    wc_features=wc_features,
    output_path="../data/world_cup_2026/world_cup_2026_quarter_final_predictions.csv",
    knockout=True
)

Shape: (4, 11)

Missing values:


match_number             0
home_team                0
away_team                0
neutral                  0
home_recent_win_rate     0
away_recent_win_rate     0
home_recent_goal_diff    0
away_recent_goal_diff    0
home_rank                0
away_rank                0
rank_difference          0
dtype: int64


Rows with missing model features: 0


,match_number,home_team,away_team,neutral,home_recent_win_rate,away_recent_win_rate,home_recent_goal_diff,away_recent_goal_diff,home_rank,away_rank,rank_difference


Saved: ../data/world_cup_2026/world_cup_2026_quarter_final_predictions.csv


,match_number,home_team,away_team,predicted_result,predicted_winner,home_rank,away_rank
0,97,France,Netherlands,home_win,France,1,7
1,98,Spain,United States,home_win,Spain,2,16
2,99,Brazil,Mexico,home_win,Brazil,6,15
3,100,Argentina,Canada,home_win,Argentina,3,30


## Semi-Final Predictions

The semi-final matchups were manually entered based on the predicted quarter-final winners.

In [12]:
semi_final_matches = [
    ("France", "Spain"),
    ("Brazil", "Argentina")
]

semi_final_fixtures = []

for i, (home_team, away_team) in enumerate(semi_final_matches, start=101):
    neutral = 0 if home_team in host_teams or away_team in host_teams else 1

    semi_final_fixtures.append({
        "match_number": i,
        "home_team": home_team,
        "away_team": away_team,
        "neutral": neutral
    })

semi_final_fixtures = pd.DataFrame(semi_final_fixtures)

semi_final_fixtures.to_csv(
    "../data/world_cup_2026/world_cup_2026_semi_final_fixtures.csv",
    index=False
)

display(semi_final_fixtures)

,match_number,home_team,away_team,neutral
0,101,France,Spain,1
1,102,Brazil,Argentina,1


In [13]:
semi_final_predictions = predict_matches(
    fixtures_df=semi_final_fixtures,
    wc_features=wc_features,
    output_path="../data/world_cup_2026/world_cup_2026_semi_final_predictions.csv",
    knockout=True
)

Shape: (2, 11)

Missing values:


match_number             0
home_team                0
away_team                0
neutral                  0
home_recent_win_rate     0
away_recent_win_rate     0
home_recent_goal_diff    0
away_recent_goal_diff    0
home_rank                0
away_rank                0
rank_difference          0
dtype: int64


Rows with missing model features: 0


,match_number,home_team,away_team,neutral,home_recent_win_rate,away_recent_win_rate,home_recent_goal_diff,away_recent_goal_diff,home_rank,away_rank,rank_difference


Saved: ../data/world_cup_2026/world_cup_2026_semi_final_predictions.csv


,match_number,home_team,away_team,predicted_result,predicted_winner,home_rank,away_rank
0,101,France,Spain,away_win,Spain,1,2
1,102,Brazil,Argentina,away_win,Argentina,6,3


## Final Prediction

The final matchup is predicted using the same model features. The predicted winner is our model's projected 2026 World Cup champion.

In [14]:
final_matches = [
    ("Spain", "Argentina")
]

final_fixtures = []

for i, (home_team, away_team) in enumerate(final_matches, start=104):
    neutral = 0 if home_team in host_teams or away_team in host_teams else 1

    final_fixtures.append({
        "match_number": i,
        "home_team": home_team,
        "away_team": away_team,
        "neutral": neutral
    })

final_fixtures = pd.DataFrame(final_fixtures)

final_fixtures.to_csv(
    "../data/world_cup_2026/world_cup_2026_final_fixtures.csv",
    index=False
)

display(final_fixtures)

,match_number,home_team,away_team,neutral
0,104,Spain,Argentina,1


In [15]:
final_predictions = predict_matches(
    fixtures_df=final_fixtures,
    wc_features=wc_features,
    output_path="../data/world_cup_2026/world_cup_2026_final_predictions.csv",
    knockout=True
)

champion = final_predictions.loc[0, "predicted_winner"]

print("Predicted 2026 World Cup Champion:", champion)

Shape: (1, 11)

Missing values:


match_number             0
home_team                0
away_team                0
neutral                  0
home_recent_win_rate     0
away_recent_win_rate     0
home_recent_goal_diff    0
away_recent_goal_diff    0
home_rank                0
away_rank                0
rank_difference          0
dtype: int64


Rows with missing model features: 0


,match_number,home_team,away_team,neutral,home_recent_win_rate,away_recent_win_rate,home_recent_goal_diff,away_recent_goal_diff,home_rank,away_rank,rank_difference


Saved: ../data/world_cup_2026/world_cup_2026_final_predictions.csv


,match_number,home_team,away_team,predicted_result,predicted_winner,home_rank,away_rank
0,104,Spain,Argentina,home_win,Spain,2,3


Predicted 2026 World Cup Champion: Spain


## Final Prediction Summary

This section combines the predicted winner from each knockout round and highlights the model's final projected champion.

In [16]:
summary_rows = []

round_outputs = [
    ("Round of 32", round_of_32_predictions),
    ("Round of 16", round_of_16_predictions),
    ("Quarter-Finals", quarter_final_predictions),
    ("Semi-Finals", semi_final_predictions),
    ("Final", final_predictions)
]

for round_name, round_df in round_outputs:
    for _, row in round_df.iterrows():
        summary_rows.append({
            "round": round_name,
            "match_number": row["match_number"],
            "home_team": row["home_team"],
            "away_team": row["away_team"],
            "predicted_result": row["predicted_result"],
            "predicted_winner": row["predicted_winner"]
        })

prediction_summary = pd.DataFrame(summary_rows)

prediction_summary.to_csv(
    "../data/world_cup_2026/world_cup_2026_prediction_summary.csv",
    index=False
)

display(prediction_summary)

print("Predicted 2026 World Cup Champion:", champion)

,round,match_number,home_team,away_team,predicted_result,predicted_winner
0,Round of 32,73,South Korea,Switzerland,away_win,Switzerland
1,Round of 32,74,Germany,Scotland,home_win,Germany
2,Round of 32,75,Netherlands,Morocco,home_win,Netherlands
3,Round of 32,76,Brazil,Japan,home_win,Brazil
4,Round of 32,77,France,Australia,home_win,France
5,Round of 32,78,Ecuador,Senegal,home_win,Ecuador
6,Round of 32,79,Mexico,Ivory Coast,home_win,Mexico
7,Round of 32,80,England,DR Congo,home_win,England
8,Round of 32,81,United States,Austria,home_win,United States
9,Round of 32,82,Belgium,Czech Republic,home_win,Belgium


Predicted 2026 World Cup Champion: Spain


## Extension Limitations

This World Cup 2026 prediction section is an extension of the main project. The model was trained on historical international matches, but the 2026 tournament has not happened yet, so these predictions cannot be validated.

The extension also uses manually collected current team features from FIFA ranking information. The recent win rate and average goal differential values are simplified proxies for team form.

Important real-world factors such as injuries, squad selection, venue details, tactics, weather, rest days, and penalty shootouts are not included.

Therefore, these predictions should be interpreted as an exploratory simulation rather than a definitive forecast.